In [1]:
import numpy as np
import scipy.stats as stats
from plotly.io import show
from sklearn.model_selection import RandomizedSearchCV, train_test_split

from skfolio import PerfMeasure, Population, RatioMeasure, RiskMeasure
from skfolio.datasets import load_sp500_dataset
from skfolio.distance import KendallDistance, PearsonDistance
from skfolio.metrics import make_scorer
from skfolio.model_selection import MultipleRandomizedCV, WalkForward, cross_val_predict
from skfolio.moments import (
    LedoitWolf,
)
from skfolio.optimization import (
    HierarchicalRiskParity,
    MeanRisk,
    SchurComplementary,
)
from skfolio.preprocessing import prices_to_returns
from skfolio.prior import EmpiricalPrior

prices = load_sp500_dataset()
X = prices_to_returns(prices)

In [2]:
# `shuffle=False` preserves chronological order, crucial for time-series data.
X_train, X_test = train_test_split(X.iloc[:, 10:], test_size=0.3, shuffle=False)

In [3]:
model = SchurComplementary(gamma=0.5)
model.fit(X_train)
print(model.weights_)

[0.03687455 0.0613539  0.0659264  0.18201864 0.03743585 0.21311991
 0.02811169 0.07392489 0.10507859 0.19615558]


In [4]:
prior = EmpiricalPrior(covariance_estimator=LedoitWolf())

population_train = Population([])
population_test = Population([])

# 20 Schur portfolios
for gamma in np.linspace(0.0, 1.0, 20):
    schur = SchurComplementary(
        gamma=gamma,
        prior_estimator=prior,
        portfolio_params={"name": f"Schur {gamma:0.2f}", "tag": "Schur"},
    )
    # Train
    ptf = schur.fit_predict(X_train)
    population_train.append(ptf)
    # Test
    ptf = schur.predict(X_test)
    population_test.append(ptf)

# HRP portfolio
hrp = HierarchicalRiskParity(prior_estimator=prior, portfolio_params={"tag": "HRP"})
# Train
ptf = hrp.fit_predict(X_train)
population_train.append(ptf)
hrp_std = ptf.standard_deviation
# Test
ptf = hrp.predict(X_test)
population_test.append(ptf)

# 20 MVO (including MVP) portfolios
mean_variance = MeanRisk(
    prior_estimator=prior,
    efficient_frontier_size=20,
    max_standard_deviation=hrp_std,
    portfolio_params={"tag": "MVO"},
)
# Train
mv_population_train = mean_variance.fit_predict(X_train)
mv_population_train[0].tag = "MVP"
population_train += mv_population_train
# Test
mv_population_test = mean_variance.predict(X_test)
mv_population_test[0].tag = "MVP"
population_test += mv_population_test

In [5]:
fig = population_train.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    hover_measures=[RatioMeasure.ANNUALIZED_SHARPE_RATIO],
    title="Training Set | MVO - HRP - Schur",
)
show(fig)

In [6]:
population_test.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    hover_measures=[RatioMeasure.ANNUALIZED_SHARPE_RATIO],
    title="Test Set | MVO - HRP - Schur",
)

In [7]:
population_train.filter(tags=["Schur", "MVP", "MVO"]).plot_composition()

In [8]:
walk_forward = WalkForward(test_size=60, train_size=252 * 3)

In [9]:
model = SchurComplementary(prior_estimator=prior)

random_search = RandomizedSearchCV(
    estimator=model,
    cv=walk_forward,
    n_jobs=-1,
    param_distributions={
        "gamma": stats.uniform(0, 1),
        "distance_estimator": [PearsonDistance(), KendallDistance()],
    },
    n_iter=10,
    scoring=make_scorer(RatioMeasure.CDAR_RATIO),
    random_state=0,
)
random_search.fit(X_train)

# Retrieve the best estimator from the search.
schur = random_search.best_estimator_
schur

,gamma,0.5288949197529045
,keep_monotonic,True
,prior_estimator,EmpiricalPrio...=LedoitWolf())
,distance_estimator,KendallDistance()
,hierarchical_clustering_estimator,None
,min_weights,0.0
,max_weights,1.0
,transaction_costs,0.0
,management_fees,0.0
,previous_weights,None
,portfolio_params,None


In [10]:
mvo = MeanRisk(prior_estimator=prior)
pred_mvo = cross_val_predict(mvo, X_test, cv=walk_forward, n_jobs=-1)
pred_mvo.name = "MVP"

pred_schur = cross_val_predict(schur, X_test, cv=walk_forward, n_jobs=-1)
pred_schur.name = "Schur"

# Combine results for easier analysis.
population = Population([pred_schur, pred_mvo])
population.plot_cumulative_returns()

In [11]:
summary = population.summary()
print(summary.loc[["Annualized Sharpe Ratio", "CDaR Ratio at 95%"]])

                          Schur     MVP
Annualized Sharpe Ratio    0.99    0.72
CDaR Ratio at 95%        0.0052  0.0032


In [12]:
X_train, X_test = train_test_split(X, test_size=0.3, shuffle=False)

cv_mc = MultipleRandomizedCV(
    walk_forward=walk_forward,
    n_subsamples=800,
    asset_subset_size=10,
    window_size=5 * 252,
    random_state=0,
)

# Generate cross-validated predictions for both models.
pred_mvo_mc = cross_val_predict(
    mvo, X_test, cv=cv_mc, n_jobs=-1, portfolio_params={"tag": "MVP"}
)

pred_schur_mc = cross_val_predict(
    schur, X_test, cv=cv_mc, n_jobs=-1, portfolio_params={"tag": "Schur"}
)

# Combine results for easier analysis.
population_mc = pred_mvo_mc + pred_schur_mc

In [13]:
population_mc.plot_distribution(
    measure_list=[RatioMeasure.ANNUALIZED_SHARPE_RATIO], tag_list=["MVP", "Schur"]
)

In [14]:
population_mc.plot_distribution(
    measure_list=[RatioMeasure.CDAR_RATIO], tag_list=["MVP", "Schur"]
)

In [15]:
for pred in [pred_mvo_mc, pred_schur_mc]:
    tag = pred[0].tag
    mean_sr = pred.measures_mean(measure=RatioMeasure.ANNUALIZED_SHARPE_RATIO)
    std_sr = pred.measures_std(measure=RatioMeasure.ANNUALIZED_SHARPE_RATIO)
    print(f"{tag}\n{'=' * len(tag)}")
    print(f"Average Sharpe Ratio: {mean_sr:0.2f}")
    print(f"Sharpe Ratio Std Dev: {std_sr:0.2f}\n")

MVP
===
Average Sharpe Ratio: 0.74
Sharpe Ratio Std Dev: 0.37

Schur
=====
Average Sharpe Ratio: 0.91
Sharpe Ratio Std Dev: 0.42



Schur portfolios tend to outperform MVP out-of-sample, exhibiting higher average Sharpe and CDaR ratios.